# Alkahest 0.8B Two-Stage SFT

Trains Stage A instruction/format preservation from `thomasjvu/alkahest-0.8b-heretic-merged`, merges it, then trains Stage B as roleplay-first false-refusal correction with a small boundary slice from the Stage A merged checkpoint. Outputs stay under `/kaggle/working/alkahest-08b-two-stage-sft`.

In [ ]:
import os, platform, shutil, subprocess
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major < 7:
            raise RuntimeError('Kaggle assigned a pre-Volta GPU. Push with --accelerator NvidiaTeslaT4.')
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    if 'pre-Volta' in str(exc):
        raise

In [ ]:
import os, subprocess, sys

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
try:
    from kaggle_secrets import UserSecretsClient
    secret_token = UserSecretsClient().get_secret('HF_TOKEN')
    if secret_token and not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = secret_token
    if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
        os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
    print('hf_secret_loaded=', bool(secret_token))
except Exception as exc:
    print('hf_secret_loaded=', False, type(exc).__name__)


packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
import transformers, peft, huggingface_hub
print('transformers=', transformers.__version__)
print('peft=', peft.__version__)
print('huggingface_hub=', huggingface_hub.__version__)

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/alkahest-ai/heretic-to-onnx.git'
REPO_REF = 'codex/kaggle-heretic-2b-run'
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-two-stage-sft')
SPLIT_DIR = WORK_DIR / 'splits'
WORK_DIR.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts/prepare_alkahest_two_stage_sft.py'),
    '--output-dir', str(SPLIT_DIR),
    '--stage-a-repeats', os.environ.get('TWO_STAGE_A_REPEATS', '18'),
    '--stage-b-boundary-repeats', os.environ.get('TWO_STAGE_B_BOUNDARY_REPEATS', '80'),
    '--stage-b-adult-repeats', os.environ.get('TWO_STAGE_B_ADULT_REPEATS', '40'),
    '--val-fraction', os.environ.get('TWO_STAGE_VAL_FRACTION', '0.10'),
])

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-two-stage-sft')
SPLIT_DIR = WORK_DIR / 'splits'
A_ADAPTER_DIR = WORK_DIR / 'stage-a-adapter'
A_MERGED_DIR = WORK_DIR / 'stage-a-merged'

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/train_alkahest_sft.py'),
    '--model-name', 'thomasjvu/alkahest-0.8b-heretic-merged',
    '--train-file', str(SPLIT_DIR / 'stage_a' / 'train.jsonl'),
    '--val-file', str(SPLIT_DIR / 'stage_a' / 'val.jsonl'),
    '--dataset-manifest', str(SPLIT_DIR / 'manifest.json'),
    '--output-dir', str(A_ADAPTER_DIR),
    '--merged-output-dir', str(A_MERGED_DIR),
    '--max-seq-length', os.environ.get('STAGE_A_MAX_SEQ_LENGTH', '768'),
    '--max-steps', os.environ.get('STAGE_A_MAX_STEPS', '80'),
    '--gradient-accumulation-steps', os.environ.get('STAGE_A_GRAD_ACCUM', '8'),
    '--learning-rate', os.environ.get('STAGE_A_LR', '4e-6'),
    '--eval-steps', '20',
    '--save-steps', '40',
]
print(' '.join(cmd))
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.check_call(cmd, env=env)

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-two-stage-sft')
SPLIT_DIR = WORK_DIR / 'splits'
A_MERGED_DIR = WORK_DIR / 'stage-a-merged'
B_ADAPTER_DIR = WORK_DIR / 'stage-b-adapter'
AB_MERGED_DIR = WORK_DIR / 'stage-ab-merged'

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/train_alkahest_sft.py'),
    '--model-name', str(A_MERGED_DIR),
    '--train-file', str(SPLIT_DIR / 'stage_b' / 'train.jsonl'),
    '--val-file', str(SPLIT_DIR / 'stage_b' / 'val.jsonl'),
    '--dataset-manifest', str(SPLIT_DIR / 'manifest.json'),
    '--output-dir', str(B_ADAPTER_DIR),
    '--merged-output-dir', str(AB_MERGED_DIR),
    '--max-seq-length', os.environ.get('STAGE_B_MAX_SEQ_LENGTH', '768'),
    '--max-steps', os.environ.get('STAGE_B_MAX_STEPS', '220'),
    '--gradient-accumulation-steps', os.environ.get('STAGE_B_GRAD_ACCUM', '8'),
    '--learning-rate', os.environ.get('STAGE_B_LR', '8e-6'),
    '--eval-steps', '15',
    '--save-steps', '45',
]
print(' '.join(cmd))
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.check_call(cmd, env=env)

In [ ]:
from pathlib import Path
import json

WORK_DIR = Path('/kaggle/working/alkahest-08b-two-stage-sft')
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(WORK_DIR), round(path.stat().st_size / 1024**2, 2), 'MB')
for report in [WORK_DIR / 'stage-a-adapter' / 'training-run.json', WORK_DIR / 'stage-b-adapter' / 'training-run.json']:
    if report.exists():
        print(report)
        print(report.read_text()[:4000])